In [1]:
import sys
import asyncio
import io

# Force Windows to use the Proactor Event Loop for async subprocesses
if sys.platform == "win32":
    asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())

# Patch Jupyter's OutStream to prevent the fileno crash during process fallback
if hasattr(sys.stderr, "fileno"):
    try:
        sys.stderr.fileno()
    except io.UnsupportedOperation:
        sys.stderr.fileno = lambda: 2

In [2]:
#create mcp ai agent to manipulate mongodb

import os 
from dotenv import load_dotenv
load_dotenv()
OPENWEATHER_API_KEY = os.getenv("WEATHER_API_KEY")
os.environ["TAVILY_API_KEY"] = os.getenv("TAVILY_API_KEY")

In [3]:
import warnings 
warnings.filterwarnings(action="ignore", category=DeprecationWarning)


In [ ]:
from langchain.chat_models import init_chat_model

model_llama = init_chat_model("llama-3.3-70b-versatile", model_provider="groq")
model_qwen = init_chat_model("qwen-2.5-32b", model_provider="groq")
model_gem = init_chat_model("gemini-3.6-flash", model_provider="google_genai")

In [5]:
#create mcp client & connect with mcp-mongodb server
from langchain_mcp_adapters.client import MultiServerMCPClient

In [ ]:
import os

mongodb_uri = os.getenv("MONGODB_URI", "mongodb://localhost:27017")

client = MultiServerMCPClient(
    {
        "mongodb": {
            "transport": "stdio",
            # Execute Node directly on the installed module binary
            "command": "node",
            "args": [
                r"C:\Users\ramma\AppData\Roaming\npm\node_modules\mongodb-mcp-server\dist\index.js"
            ],
            "env": {
                "PATH": os.environ.get("PATH", ""),
                "MDB_MCP_CONNECTION_STRING": mongodb_uri,
            },
        }
    }
)

In [7]:
import os
import subprocess

# Set your connection string or fallback
os.environ["MDB_MCP_CONNECTION_STRING"] = os.getenv(
    "MONGODB_URI", "mongodb://localhost:27017"
)

# Run via npx directly and capture stderr
cmd = ["npx.cmd", "-y", "mongodb-mcp-server@latest"]
res = subprocess.run(
    cmd,
    capture_output=True,
    text=True,
    env=os.environ,
    shell=True,
)

print("--- STDOUT ---")
print(res.stdout)
print("--- STDERR ---")
print(res.stderr)

--- STDOUT ---

--- STDERR ---
node:internal/modules/esm/resolve:229
    throw new ERR_UNSUPPORTED_DIR_IMPORT(path, basePath, String(resolved));
          ^

Error [ERR_UNSUPPORTED_DIR_IMPORT]: Directory import 'C:\Users\ramma\AppData\Local\npm-cache\_npx\57c35de36e7d43c4\node_modules\zod\v4' is not supported resolving ES modules imported from C:\Users\ramma\AppData\Local\npm-cache\_npx\57c35de36e7d43c4\node_modules\@modelcontextprotocol\core\dist\schemas-DscsoNZN.mjs
    at finalizeResolution (node:internal/modules/esm/resolve:229:11)
    at moduleResolve (node:internal/modules/esm/resolve:846:10)
    at defaultResolve (node:internal/modules/esm/resolve:973:11)
    at #cachedDefaultResolve (node:internal/modules/esm/loader:708:20)
    at #resolveAndMaybeBlockOnLoaderThread (node:internal/modules/esm/loader:728:38)
    at ModuleLoader.resolveSync (node:internal/modules/esm/loader:766:56)
    at #resolve (node:internal/modules/esm/loader:690:17)
    at ModuleLoader.getOrCreateModuleJob 

In [8]:
tools = await client.get_tools()

In [9]:
for tool in tools:
    print(tool.name)
    print(tool.description)
    print(tool.args_schema)

aggregate-db
Run an aggregation against a MongoDB database
{'type': 'object', '$schema': 'https://json-schema.org/draft/2020-12/schema', 'properties': {'connectionId': {'type': 'string', 'description': 'The connection to run the operation against. Use the id returned by one of the connect tools, or "preconfigured" to use the connection string the server was configured with.'}, 'database': {'type': 'string', 'description': 'Database name'}, 'pipeline': {'minItems': 1, 'type': 'array', 'items': {'type': 'object', 'properties': {}, 'additionalProperties': True}, 'description': 'An array of aggregation stages to execute. The first stage must be a database-level aggregation stage (one of `$changeStream`, `$currentOp`, `$documents`, `$listLocalSessions`, `$queryStats`). https://www.mongodb.com/docs/manual/reference/mql/aggregation-stages/#db.aggregate---stages'}, 'responseBytesLimit': {'default': 1048576, 'description': "The maximum number of bytes to return in the response. This value is ca

In [10]:
prompt = """You are a helpful MCP assistant. Use tolls for answering query related to mongoDB database.
For other queries you can answer on your own. """

In [11]:
#create agent with mongodb tools
from langchain.agents import create_agent
agent = create_agent(
    model= model_gem,
    tools = tools,
    system_prompt = prompt
)

In [20]:
#test the agent
user_query = """How many documents are there? Show each documents name field and ID
    Database: mcpDB, Collection name: students"""

In [13]:
#insert new student

In [14]:
#find a student by name

In [15]:
#update an existing student

In [16]:
#delete a student

In [17]:
from langchain.messages import HumanMessage

In [21]:
try: 
    response = await agent.ainvoke({
        "messages":[
            HumanMessage(content = user_query)
        ]
    })
except Exception as e:
    raise RuntimeError(f"Error, {e}")

Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is not supported in schema, ignoring
Key '$schema' is not supported in schema, ignoring
Key 'additionalProperties' is

In [22]:
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

How many documents are there? Show each documents name field and ID
    Database: mcpDB, Collection name: students
================================== Ai Message ==================================

[]
Tool Calls:
  count (call_15571463781101277025)
 Call ID: call_15571463781101277025
  Args:
    collection: students
    connectionId: preconfigured
    database: mcpDB
  find (call_15571463781101274178)
 Call ID: call_15571463781101274178
  Args:
    collection: students
    projection: {'name': 1, '_id': 1}
    connectionId: preconfigured
    database: mcpDB
================================= Tool Message =================================
Name: count

[{'type': 'text', 'text': 'Found 1 documents in the collection "students".', 'id': 'lc_c53f4912-7e30-401a-87d4-729c14899256'}]
================================= Tool Message =================================
Name: find

[{'type': 'text', 'text': 'Query on colle